# Predictive 4th Down Model — Playcaller Decision

**Goal:** Predict what decision a playcaller will actually make on 4th down (go-for-it, punt, field goal), then compare each playcaller's predicted decisions against the prescriptive EPA model's optimal recommendations.

**Pipeline:**
1. Load & filter data
2. Select pre-snap features; set target = actual coach decision
3. Train/test split (season-based)
4. Train XGBoost to predict actual decisions
5. Recompute prescriptive optimal labels
6. Per-playcaller agreement analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, log_loss, accuracy_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform, loguniform
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
print('Libraries loaded.')

## 1. Load & Filter Data

In [ ]:
df = pd.read_csv('../encoded_fourth_downs.csv', low_memory=False)
print(f'Raw rows: {len(df):,}')
print(f'Seasons: {sorted(df["season"].unique())}')

# Reconstruct actual coach decision
if 'play_type' in df.columns:
    play_type_map = {'run': 'go', 'pass': 'go', 'punt': 'punt', 'field_goal': 'field_goal'}
    df['decision'] = df['play_type'].map(play_type_map)
else:
    def get_decision(row):
        if row.get('play_type_pass', 0) == 1 or row.get('play_type_run', 0) == 1:
            return 'go'
        elif row.get('play_type_punt', 0) == 1:
            return 'punt'
        elif row.get('field_goal_attempt', 0) == 1:
            return 'field_goal'
        return np.nan
    df['decision'] = df.apply(get_decision, axis=1)

df_clean = df.dropna(subset=['decision']).copy()
df_clean = df_clean[df_clean['decision'].isin(['go', 'punt', 'field_goal'])]

print(f'\nUsable 4th down plays: {len(df_clean):,}')
print(df_clean['decision'].value_counts())

## 2. Features & Target

Target is the **actual coach decision**. Same pre-snap feature set as the prescriptive model.

In [ ]:
pre_snap_features = [
    'yardline_100', 'ydstogo', 'score_differential',
    'game_seconds_remaining', 'half_seconds_remaining', 'qtr',
    'posteam_timeouts_remaining', 'defteam_timeouts_remaining',
    'goal_to_go', 'shotgun', 'no_huddle', 'home_is_posteam',
]

pregame_epa_features = [
    'no_score_prob', 'opp_fg_prob', 'opp_td_prob', 'fg_prob', 'td_prob',
]

playcaller_cols = [c for c in df_clean.columns if c.startswith('playcaller_')]

df_clean['is_dynamic_era'] = (df_clean['season'] >= 2023).astype(int)

season_dummies = pd.get_dummies(df_clean['season'], prefix='season', drop_first=True)
df_clean = pd.concat([df_clean, season_dummies], axis=1)
season_feat_cols = list(season_dummies.columns)

all_features = (
    pre_snap_features + pregame_epa_features +
    playcaller_cols + ['is_dynamic_era'] + season_feat_cols
)
all_features = [f for f in all_features if f in df_clean.columns]

X = df_clean[all_features].copy()
y = df_clean['decision'].copy()   # <-- actual coach decision

bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)

mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]

print(f'Features: {len(all_features)}')
print(f'Dataset:  {len(X):,} plays')
print(f'Target distribution:\n{y.value_counts()}')

## 3. Train/Test Split — Season-Based

2018-2023 train, 2024 test.

In [ ]:
TEST_SEASON = 2024
season_col = df_clean.loc[mask, 'season']

train_idx = season_col[season_col < TEST_SEASON].index
test_idx  = season_col[season_col == TEST_SEASON].index

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train seasons: {sorted(season_col[train_idx].unique())}')
print(f'Test season:   {sorted(season_col[test_idx].unique())}')

## 4. Train XGBoost to Predict Actual Coach Decisions

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

param_dist = {
    'n_estimators':     randint(100, 600),
    'max_depth':        randint(3, 10),
    'learning_rate':    loguniform(0.01, 0.3),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 20),
    'gamma':            uniform(0, 5),
    'reg_alpha':        loguniform(0.001, 10),
    'reg_lambda':       loguniform(0.001, 10),
}

xgb_base = xgb.XGBClassifier(
    eval_metric='mlogloss', random_state=42, n_jobs=-1, verbosity=0
)

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=30,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring={'accuracy': 'accuracy', 'neg_log_loss': 'neg_log_loss'},
    refit='neg_log_loss',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

search.fit(X_train, y_train_enc)
predictive_model = search.best_estimator_

test_preds_enc = predictive_model.predict(X_test)
test_preds     = le.inverse_transform(test_preds_enc)
test_probs     = predictive_model.predict_proba(X_test)

print('\n=== Best Parameters ===')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

print('\n=== Test Set Performance (predicting actual coach decision) ===')
print(classification_report(y_test, test_preds, target_names=le.classes_))
print(f'Accuracy: {accuracy_score(y_test, test_preds):.4f}')
print(f'Log loss: {log_loss(y_test_enc, test_probs):.4f}')

## 5. Recompute Prescriptive Optimal Labels

Same bucketing logic as the prescriptive notebook. Requires `epa` column.

In [ ]:
df_prescriptive = df_clean.dropna(subset=['epa']).copy()

df_prescriptive['ydstogo_bin'] = pd.cut(
    df_prescriptive['ydstogo'], bins=[0, 1, 3, 6, 10, 99],
    labels=['1', '2-3', '4-6', '7-10', '10+']
)
df_prescriptive['yardline_bin'] = pd.cut(
    df_prescriptive['yardline_100'], bins=[0, 20, 40, 60, 80, 100],
    labels=['opp_red_zone', 'opp_40', 'midfield', 'own_40', 'own_end']
)
df_prescriptive['score_diff_bin'] = pd.cut(
    df_prescriptive['score_differential'],
    bins=[-100, -14, -7, -3, 3, 7, 14, 100],
    labels=['down_14+', 'down_8-14', 'down_1-7', 'close', 'up_1-7', 'up_8-14', 'up_14+']
)
df_prescriptive['kickoff_era'] = df_prescriptive['season'].apply(
    lambda s: 'dynamic' if s >= 2023 else 'traditional'
)

group_cols = ['ydstogo_bin', 'yardline_bin', 'score_diff_bin', 'kickoff_era']

epa_pivot = (
    df_prescriptive
    .groupby(group_cols + ['decision'])['epa']
    .mean()
    .reset_index()
    .rename(columns={'epa': 'mean_epa'})
    .pivot_table(index=group_cols, columns='decision', values='mean_epa')
    .reset_index()
)
epa_pivot.columns.name = None

for col in ['go', 'punt', 'field_goal']:
    if col not in epa_pivot.columns:
        epa_pivot[col] = np.nan

epa_pivot['optimal_decision'] = epa_pivot[['go', 'punt', 'field_goal']].idxmax(axis=1)

df_prescriptive = df_prescriptive.merge(
    epa_pivot[group_cols + ['optimal_decision']], on=group_cols, how='left'
).dropna(subset=['optimal_decision'])

print(f'Rows with prescriptive label: {len(df_prescriptive):,}')
print(df_prescriptive['optimal_decision'].value_counts())

## 6. Per-Playcaller Agreement Analysis

In [ ]:
# Score all plays that have complete features
X_full = df_prescriptive[all_features].copy()
X_full[X_full.select_dtypes(include='bool').columns] = \
    X_full.select_dtypes(include='bool').astype(int)

full_mask = X_full.notna().all(axis=1)
predicted_enc = predictive_model.predict(X_full[full_mask])
predicted     = le.inverse_transform(predicted_enc)

df_scored = df_prescriptive[full_mask].copy()
df_scored['predicted_decision'] = predicted

# Identify each row's playcaller from the one-hot columns
pc_cols_present = [c for c in playcaller_cols if c in df_scored.columns]
df_scored['playcaller'] = df_scored[pc_cols_present].apply(
    lambda row: next(
        (col.replace('playcaller_', '') for col in pc_cols_present if row[col] == 1),
        'Unknown'
    ),
    axis=1
)

# Agreement flags
df_scored['actual_matches_optimal']    = (df_scored['decision']           == df_scored['optimal_decision']).astype(int)
df_scored['predicted_matches_optimal'] = (df_scored['predicted_decision'] == df_scored['optimal_decision']).astype(int)

overall_actual_agree    = df_scored['actual_matches_optimal'].mean()
overall_predicted_agree = df_scored['predicted_matches_optimal'].mean()
print(f'League-wide actual agreement with optimal:    {overall_actual_agree:.1%}')
print(f'League-wide predicted agreement with optimal: {overall_predicted_agree:.1%}')

In [ ]:
# Per-playcaller stats (min 20 plays)
MIN_PLAYS = 20

playcaller_stats = (
    df_scored
    .groupby('playcaller')
    .agg(
        n_plays              = ('decision', 'count'),
        actual_agree_rate    = ('actual_matches_optimal', 'mean'),
        predicted_agree_rate = ('predicted_matches_optimal', 'mean'),
    )
    .query(f'n_plays >= {MIN_PLAYS}')
    .sort_values('actual_agree_rate', ascending=False)
    .round(3)
)

print(f'Playcallers with >= {MIN_PLAYS} plays: {len(playcaller_stats)}')
print(playcaller_stats.to_string())

In [ ]:
# Bar chart: actual vs predicted agreement per playcaller (top 25 by volume)
plot_df = playcaller_stats.nlargest(25, 'n_plays').sort_values('actual_agree_rate')

fig, ax = plt.subplots(figsize=(10, 8))
y_pos      = np.arange(len(plot_df))
bar_height = 0.35

ax.barh(y_pos - bar_height/2, plot_df['actual_agree_rate'],    bar_height,
        label='Actual coach decision', color='steelblue')
ax.barh(y_pos + bar_height/2, plot_df['predicted_agree_rate'], bar_height,
        label='Predictive model', color='darkorange', alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df.index, fontsize=9)
ax.axvline(overall_actual_agree,    color='steelblue',  linestyle='--', alpha=0.5,
           label=f'League avg actual ({overall_actual_agree:.1%})')
ax.axvline(overall_predicted_agree, color='darkorange', linestyle='--', alpha=0.5,
           label=f'League avg predicted ({overall_predicted_agree:.1%})')
ax.set_xlabel('Agreement Rate with Prescriptive (Optimal EPA) Model')
ax.set_title('Playcaller Agreement with Optimal EPA Decision\n(Top 25 by volume — actual vs predicted)')
ax.legend(loc='lower right', fontsize=8)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig('playcaller_agreement.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: playcaller_agreement.png')

In [ ]:
# Scatter: actual vs predicted agreement — outliers = playcallers the model misjudges
playcaller_stats['diag_gap'] = (
    playcaller_stats['predicted_agree_rate'] - playcaller_stats['actual_agree_rate']
).abs()

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(
    playcaller_stats['actual_agree_rate'],
    playcaller_stats['predicted_agree_rate'],
    s=playcaller_stats['n_plays'] / 5,
    alpha=0.7, color='steelblue', edgecolors='k', linewidths=0.4
)

lims = [0.3, 1.0]
ax.plot(lims, lims, 'k--', alpha=0.4, linewidth=1)

for pc in playcaller_stats.nlargest(5, 'diag_gap').index:
    row = playcaller_stats.loc[pc]
    ax.annotate(pc, (row['actual_agree_rate'], row['predicted_agree_rate']),
                fontsize=7, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Actual Coach Agreement with Prescriptive')
ax.set_ylabel('Predicted Agreement with Prescriptive')
ax.set_title('Actual vs Predicted Agreement with Optimal EPA Decision\n(bubble size prop to play volume)')
plt.tight_layout()
plt.savefig('playcaller_agreement_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: playcaller_agreement_scatter.png')

In [ ]:
print('=== Most Optimal Playcallers ===')
print(playcaller_stats['actual_agree_rate'].nlargest(5).to_string())

print('\n=== Least Optimal Playcallers ===')
print(playcaller_stats['actual_agree_rate'].nsmallest(5).to_string())